# ViNewsQA FullFT — Qwen2.5-3B (Full Fine-tuning, RTX 4090)

Pipeline:
1) Token profiling → chọn `max_seq_length`.
2) Full fine-tune trên `train_ViNewsQA.json` với early stopping dựa trên `dev_ViNewsQA.json`.
3) Evaluate trên `test_ViNewsQA.json` (1987 samples) bằng subprocess batch generation (EM/F1 + BERTScore).

RTX 4090 (~23GB): micro=1, accum=8 (effective=8) — max an toàn cho 3B FullFT. Paths/W&B dùng suffix `rtx4090`.
OOM → giữ micro=1; giảm `max_seq_length` hoặc tắt WANDB_WATCH nếu bật nhầm.

W&B:
- Bật NVML đo `gpu/power_watts`, `gpu/energy_wh`, peak VRAM + ETA trong quá trình train.
- Tắt generation table để ETA ổn định (`WANDB_GEN_SAMPLES=0`).

Chạy notebook từ đầu (run all cells).

In [ ]:
# Install full fine-tune stack (similar to LoRA notebook)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

# Base training stack
!pip install transformers trl accelerate bitsandbytes xformers datasets safetensors scikit-learn nvidia-ml-py wandb bert-score --no-cache-dir

# Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth_zoo --no-cache-dir

# Pin PEFT last (avoid Unsloth changing API unexpectedly)
!pip install "peft==0.19.1" --no-deps --force-reinstall --no-cache-dir

print("Package install step done.")

# Quick import sanity check
import importlib
for pkg in ["torch", "transformers", "datasets", "trl", "unsloth", "peft", "accelerate"]:
    importlib.import_module(pkg)
    print("OK", pkg)


In [ ]:
import gc
import json
import math
import os
import re
import string
import unicodedata
from pathlib import Path

import torch
from datasets import Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

# ------------------------------
# Constants
# ------------------------------
NOTEBOOK_VERSION = "FULLFT_3B_RTX4090_V1"
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

DATASET_ROOT = Path("../../")
TRAIN_JSON_PATH = DATASET_ROOT / "train_ViNewsQA.json"
DEV_JSON_PATH = DATASET_ROOT / "dev_ViNewsQA.json"
TEST_JSON_PATH = DATASET_ROOT / "test_ViNewsQA.json"

PROFILING_CONFIG_PATH = "profiling_config_vinewsqa_3b_fullft_rtx4090.json"
RESULTS_FULLFT_TEST_PATH = "results_fullft_3b_test_rtx4090.json"
# Merge với LoRA 4090 compare (không đè file cũ).
COMPARE_EVAL_PATH = Path("../LoRa/eval_compare_adapters_vinewsqa_3b_test_rtx4090.json")
MERGED_COMPARE_OUT = "eval_compare_adapters_vinewsqa_3b_test_with_fullft_rtx4090.json"
BERTSCORE_OUT_PATH = "eval_compare_adapters_vinewsqa_3b_test_with_fullft_bertscore_rtx4090.json"

SYSTEM_PROMPT = (
    "Bạn là hệ thống hỏi-đáp trích xuất tiếng Việt. Chỉ trả lời bằng một cụm từ xuất hiện "
    "nguyên văn trong đoạn văn, không giải thích và không thêm tiền tố.\n\n"
    "Đoạn văn:\n{context}"
)

PREFIX_RE = re.compile(r"^(đáp án|answer|câu trả lời)\\s*[:\\-]?\\s*", re.IGNORECASE)

MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 64
INFER_BATCH_SIZE = 16  # RTX 4090: nhanh hơn (was 8); OOM infer → 8

EVAL_SPLIT = "test"  # fullft eval trên test
EXPECTED_TEST_SIZE = 1987
EVAL_STEPS = 200
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 5

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

USE_EARLY_STOPPING = True
MAX_TRAIN_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# RTX 4090 ~23GB | 3B FullFT BF16 — micro=1 là max an toàn; effective_batch=8.
# Không tăng micro mặc định (micro=2 dễ OOM).
FULLFT_MICRO_BATCH = 1
FULLFT_GRAD_ACCUM = 8

TRAIN_COMMON = dict(
    # Full FT: LR thấp hơn LoRA; 3B dùng 1e-5 (ổn định hơn 2e-5).
    per_device_train_batch_size=FULLFT_MICRO_BATCH,
    gradient_accumulation_steps=FULLFT_GRAD_ACCUM,
    warmup_ratio=0.05,
    num_train_epochs=MAX_TRAIN_EPOCHS,
    learning_rate=1e-5,
    max_grad_norm=1.0,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
)

FULLFT_OUTPUT_DIR = "outputs_vinewsqa_3b_fullft_rtx4090"
FULLFT_SAVE_PATH = Path("qwen2.5-3b-instruct-fullft-vinewsqa-rtx4090")

# ------------------------------
# W&B (NVML hardware tracking)
# ------------------------------
WANDB_ENABLED = True
WANDB_PROJECT = "qwen-3b-finetuning"
WANDB_ENTITY = None
WANDB_GROUP = "vinewsqa-3b-fullft-rtx4090"
WANDB_MODE = "online"  # online | offline | disabled

WANDB_WATCH_ENABLED = False
WANDB_HW_ENABLED = True
WANDB_HW_POLL_INTERVAL = 5.0
WANDB_GPU_INDEX = 0

# Không cần generation table để tránh ETA phình.
WANDB_GEN_SAMPLES = 0

# Nếu tắt W&B hoàn toàn, vẫn train bằng report_to="none".

# ------------------------------
# Run flags
# ------------------------------
RUN_TRAINING = True
RUN_METRIC_EVAL = True
RUN_BERTSCORE_EVAL = True
BERTSCORE_MODEL = "bert-base-multilingual-cased"
BERTSCORE_BATCH_SIZE = 64

# Resume (optional)
RESUME_TRAINING = False
RESUME_CHECKPOINT = None
RESUME_WANDB_RUN_ID = None

# TQDM bar format
TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"


In [ ]:
def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def build_messages(sample, for_inference: bool = False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=sample["context"])},
        {"role": "user", "content": sample["question"]},
    ]
    if not for_inference:
        messages.append({"role": "assistant", "content": sample["answer"]})
    return messages


def sample_to_train_text(sample, tokenizer):
    return tokenizer.apply_chat_template(
        build_messages(sample, for_inference=False),
        tokenize=False,
        add_generation_prompt=False,
    )


def load_tokenizer(model_path: str = BASE_MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.chat_template is None:
        raise RuntimeError(f"Tokenizer {model_path} has no chat template.")
    return tokenizer


def load_squad11_split(path: Path):
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)

    if payload.get("version") != "1.1" or not isinstance(payload.get("data"), list):
        raise ValueError(f"{path} is not a SQuAD 1.1 object.")

    samples = []
    dropped = 0

    for article in payload["data"]:
        title = str(article.get("title") or "")
        for paragraph in article.get("paragraphs") or []:
            context = str(paragraph.get("context") or "").strip()
            for qa in paragraph.get("qas") or []:
                question = str(qa.get("question") or "").strip()
                gold_answers = []
                for answer in qa.get("answers") or []:
                    text = str(answer.get("text") or "").strip()
                    if text and text not in gold_answers:
                        gold_answers.append(text)

                if not context or not question or not gold_answers:
                    dropped += 1
                    continue

                samples.append(
                    {
                        "id": str(qa.get("id") or ""),
                        "title": title,
                        "context": context,
                        "question": question,
                        "answer": gold_answers[0],
                        "gold_answers": gold_answers,
                    }
                )

    if not samples:
        raise RuntimeError(f"No valid labeled samples in {path}.")

    ids = [row["id"] for row in samples]
    if any(not value for value in ids) or len(ids) != len(set(ids)):
        raise RuntimeError(f"{path} contains empty or duplicate QA ids.")

    print(f"{path.name}: {len(samples)} samples | dropped={dropped}", flush=True)
    return samples


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())


def compute_em(prediction: str, truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(truth))


def compute_f1(prediction: str, truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(truth).split()
    if not pred_tokens and not truth_tokens:
        return 1.0
    if not pred_tokens or not truth_tokens:
        return 0.0

    from collections import Counter

    overlap = sum((Counter(pred_tokens) & Counter(truth_tokens)).values())
    if not overlap:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)


def score_prediction(prediction: str, gold_answers: list[str]):
    return (
        max(compute_em(prediction, gold) for gold in gold_answers),
        max(compute_f1(prediction, gold) for gold in gold_answers),
    )


# ------------------------------
# Load dataset splits
# ------------------------------
train_samples = load_squad11_split(TRAIN_JSON_PATH)
dev_samples = load_squad11_split(DEV_JSON_PATH)
test_samples = load_squad11_split(TEST_JSON_PATH)

if len(test_samples) != EXPECTED_TEST_SIZE:
    raise RuntimeError(f"Expected {EXPECTED_TEST_SIZE} test samples, got {len(test_samples)}")

if EVAL_SPLIT == "test":
    eval_samples = test_samples
elif EVAL_SPLIT == "dev":
    eval_samples = dev_samples
else:
    raise ValueError(f"Unsupported EVAL_SPLIT={EVAL_SPLIT!r}")

print(f"Early stopping split: dev ({len(dev_samples)})")
print(f"Metric evaluation split: {EVAL_SPLIT} ({len(eval_samples)})")

# Keep dev set for early stopping evaluation inside training


In [ ]:
def compute_max_seq_length(samples, tokenizer, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = [
        len(tokenizer.encode(sample_to_train_text(sample, tokenizer)))
        for sample in tqdm(samples, desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    ]
    lengths.sort()
    n = len(lengths)
    stats = {
        "min": lengths[0],
        "p50": lengths[n // 2],
        "p95": lengths[min(n - 1, int(n * 0.95))],
        "p99": lengths[min(n - 1, int(n * 0.99))],
        "max": lengths[-1],
    }

    chosen = max(min(math.ceil(stats["p99"] * 1.05), cap), min_len)
    chosen = min(cap, ((chosen + 255) // 256) * 256)

    stats["chosen_max_seq_length"] = chosen
    stats["truncated_samples"] = sum(length > chosen for length in lengths)
    stats["truncated_pct"] = round(100 * stats["truncated_samples"] / n, 3)

    return chosen, stats


tokenizer_prof = load_tokenizer()
max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)

with open(PROFILING_CONFIG_PATH, "w", encoding="utf-8") as handle:
    json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats}, handle, indent=2)

print("max_seq_length =", max_seq_length)
print(length_stats)

del tokenizer_prof
clear_gpu()


In [ ]:
tokenizer_fmt = load_tokenizer()


def formatting_prompts_func(examples):
    texts = []
    for context, question, answer in zip(examples["context"], examples["question"], examples["answer"]):
        sample = {"context": context, "question": question, "answer": answer}
        texts.append(
            tokenizer_fmt.apply_chat_template(
                build_messages(sample, for_inference=False),
                tokenize=False,
                add_generation_prompt=False,
            )
        )
    return {"text": texts}


train_hf = Dataset.from_list(train_samples)
dataset = train_hf.map(formatting_prompts_func, batched=True, remove_columns=train_hf.column_names)

dev_hf = Dataset.from_list(dev_samples)
eval_dataset = dev_hf.map(formatting_prompts_func, batched=True, remove_columns=dev_hf.column_names)

print(f"Train dataset: {len(dataset)} | Dev eval: {len(eval_dataset)}")
clear_gpu()


In [ ]:
def resolve_resume_checkpoint(output_dir: str | Path):
    """Pick latest checkpoint-* inside output_dir."""
    output_dir = Path(output_dir)
    if RESUME_CHECKPOINT:
        path = Path(RESUME_CHECKPOINT)
        if not path.exists():
            raise FileNotFoundError(f"RESUME_CHECKPOINT not found: {path}")
        return str(path)

    if RESUME_TRAINING:
        ckpts = sorted(output_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]), reverse=True)
        if ckpts:
            return str(ckpts[0])
        print(f"[resume] No checkpoint in {output_dir} — training from scratch.", flush=True)

    return None


def is_valid_full_model_dir(model_dir, min_bytes=500_000_000):
    """True nếu thư mục chứa full model hợp lệ (config + weights đủ lớn)."""
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").is_file():
        return False
    weight = None
    for name in ("model.safetensors", "pytorch_model.bin"):
        candidate = model_dir / name
        if candidate.is_file():
            weight = candidate
            break
    if weight is None:
        shards = sorted(model_dir.glob("model-*-of-*.safetensors"))
        if shards:
            weight = shards[0]
    if weight is None:
        return False
    total_size = sum(p.stat().st_size for p in model_dir.glob("model*.safetensors"))
    total_size = total_size or weight.stat().st_size
    if total_size < min_bytes:
        print(f"[eval] Invalid full model {model_dir}: {total_size / 1e6:.1f} MB (too small)", flush=True)
        return False
    print(f"[eval] OK full model {model_dir} | weights ~{total_size / 1e9:.2f} GB", flush=True)
    return True


def resolve_best_fullft_checkpoint(output_dir, save_path=None):
    """Ưu tiên best_model_checkpoint từ trainer_state; fallback checkpoint hợp lệ mới nhất."""
    output_dir = Path(output_dir)
    save_path = Path(save_path or FULLFT_SAVE_PATH)

    state_file = output_dir / "trainer_state.json"
    if state_file.is_file():
        state = json.loads(state_file.read_text(encoding="utf-8"))
        best = state.get("best_model_checkpoint")
        if best and is_valid_full_model_dir(best):
            print(f"[fullft] best checkpoint from trainer_state: {best}", flush=True)
            return str(best)

        best_step, best_loss = None, float("inf")
        for row in state.get("log_history", []):
            if "eval_loss" in row and row["eval_loss"] < best_loss:
                best_loss = row["eval_loss"]
                best_step = row.get("step")
        if best_step is not None:
            candidate = output_dir / f"checkpoint-{best_step}"
            if is_valid_full_model_dir(candidate):
                print(f"[fullft] best checkpoint from log_history: {candidate} (eval_loss={best_loss:.4f})", flush=True)
                return str(candidate)

    ckpts = sorted(output_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]), reverse=True)
    for ckpt in ckpts:
        if is_valid_full_model_dir(ckpt):
            print(f"[fullft] fallback checkpoint: {ckpt}", flush=True)
            return str(ckpt)

    if is_valid_full_model_dir(save_path):
        return str(save_path)

    raise FileNotFoundError(
        f"No valid full model in {save_path} or {output_dir}/checkpoint-*. Retrain or copy a good checkpoint."
    )


def copy_full_model_dir(src_dir, dst_dir):
    import shutil

    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for path in src_dir.iterdir():
        if path.is_file():
            shutil.copy2(path, dst_dir / path.name)
    print(f"[fullft] copied model {src_dir} -> {dst_dir}", flush=True)


class TrainProgressCallback(__import__("transformers").TrainerCallback):
    """Ép hiện tiến độ step/total dạng n/total."""

    def __init__(self):
        self.pbar = None

    def on_train_begin(self, args, state, control, **kwargs):
        total = int(state.max_steps or 0)
        if total <= 0:
            return
        from tqdm import tqdm

        self.pbar = tqdm(
            total=total,
            desc="Train",
            unit="step",
            bar_format=TQDM_BAR,
            dynamic_ncols=True,
            mininterval=0.5,
        )
        print(f"[Train] progress bar ON | 0/{total} steps", flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        self.pbar.n = int(state.global_step)
        self.pbar.refresh()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.pbar is None or not logs:
            return
        postfix = {}
        for key in ("loss", "eval_loss", "learning_rate", "epoch"):
            if key in logs and logs[key] is not None:
                val = logs[key]
                postfix[key] = f"{val:.4f}" if isinstance(val, float) else val
        if postfix:
            self.pbar.set_postfix(postfix, refresh=True)

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar is not None:
            self.pbar.n = int(state.global_step or self.pbar.total or 0)
            self.pbar.refresh()
            self.pbar.close()
            self.pbar = None


def train_fullft_one_run():
    # Local import để tránh ảnh hưởng kernel.
    import inspect
    import sys

    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer
    try:
        from trl import SFTConfig
    except ImportError:
        SFTConfig = None

    from transformers import EarlyStoppingCallback, TrainerCallback, TrainingArguments

    # W&B callbacks/utilities (reuse file from LoRa notebook)
    LORA_DIR = (Path("../LoRa")).resolve()
    if str(LORA_DIR) not in sys.path:
        sys.path.append(str(LORA_DIR))

    from wandb_integration import (
        ensure_wandb_ready,
        init_run,
        finish_run,
        build_wandb_config,
        log_train_summary,
        upload_dataset_artifact,
        WandbMetricsCallback,
        WandbHardwareCallback,
    )

    variant = {"name": "fullft", "save_path": FULLFT_SAVE_PATH, "output_dir": FULLFT_OUTPUT_DIR}
    save_path = Path(FULLFT_SAVE_PATH)
    output_dir = Path(FULLFT_OUTPUT_DIR)

    _wandb_run = None
    _use_wandb = WANDB_ENABLED

    if _use_wandb:
        _use_wandb = ensure_wandb_ready(
            mode=WANDB_MODE,
            notebook_name="train_qwen2.5_fullft_unsloth_vinewsqa_3b.ipynb",
        )

    _resume_ckpt = resolve_resume_checkpoint(output_dir)
    if _use_wandb:
        # Init W&B must happen trước khi trainer được tạo.
        _wandb_cfg = build_wandb_config(
            variant=variant,
            base_model=BASE_MODEL_NAME,
            max_seq_length=max_seq_length,
            train_common=TRAIN_COMMON,
            dataset_sizes={"train": len(dataset), "dev": len(eval_dataset) if eval_dataset is not None else 0},
            load_in_4bit=False,
            bf16=is_bfloat16_supported(),
            target_modules=TARGET_MODULES,
        )
        _wandb_run = init_run(
            variant=variant,
            config=_wandb_cfg,
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            group=WANDB_GROUP,
            tags=["3b", "qwen2.5-3b", "fullft", "rtx4090", "vinewsqa"],
            resume_run_id=RESUME_WANDB_RUN_ID if _resume_ckpt else None,
        )
        if _use_wandb:
            try:
                upload_dataset_artifact(_wandb_run, PROFILING_CONFIG_PATH)
            except Exception as exc:
                print(f"[wandb] upload_dataset_artifact skipped: {exc}", flush=True)

    clear_gpu()
    print(">>> TRAIN_FIX_V7_PICKLE <<<")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=False,
        load_in_8bit=False,
    )
    model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            pass

    eval_on = USE_EARLY_STOPPING and eval_dataset is not None

    train_hparams = TRAIN_COMMON
    print(
        f"Batch: micro={train_hparams['per_device_train_batch_size']} "
        f"accum={train_hparams['gradient_accumulation_steps']} "
        f"effective={train_hparams['per_device_train_batch_size'] * train_hparams['gradient_accumulation_steps']} "
        f"| lr={train_hparams['learning_rate']} | save={save_path} | ckpt={output_dir}",
        flush=True,
    )

    common_args = dict(
        **train_hparams,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        output_dir=str(output_dir),
        disable_tqdm=False,
        logging_strategy="steps",
        logging_steps=SAVE_STEPS,
        logging_first_step=False,
        log_level="error",
        log_level_replica="error",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="wandb" if _use_wandb else "none",
        run_name=f"vinewsqa-3b-fullft-lr{train_hparams['learning_rate']:.0e}-rtx4090",
        dataloader_num_workers=0,
        save_safetensors=True,
    )

    callbacks = [TrainProgressCallback()]
    if _use_wandb:
        callbacks.append(WandbMetricsCallback())
        if WANDB_HW_ENABLED:
            callbacks.append(
                WandbHardwareCallback(
                    gpu_index=WANDB_GPU_INDEX,
                    poll_interval_s=WANDB_HW_POLL_INTERVAL,
                )
            )

    args_cls = SFTConfig if SFTConfig is not None else TrainingArguments
    args_params = inspect.signature(args_cls.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in args_params else "evaluation_strategy"

    if eval_on:
        common_args.update({
            eval_key: "steps",
            "eval_steps": EVAL_STEPS,
            "load_best_model_at_end": True,
            "metric_for_best_model": "eval_loss",
            "greater_is_better": False,
        })
        callbacks.append(
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
            )
        )
    else:
        common_args[eval_key] = "no"

    if SFTConfig is not None:
        for key, value in (
            ("max_seq_length", max_seq_length),
            ("dataset_text_field", "text"),
            ("packing", False),
            ("dataset_num_proc", 1),
        ):
            if key in args_params:
                common_args[key] = value

    filtered_args = {key: value for key, value in common_args.items() if key in args_params or key == "output_dir"}
    training_config = args_cls(**filtered_args)

    trainer_kwargs = dict(
        model=model,
        train_dataset=dataset,
        eval_dataset=eval_dataset if eval_on else None,
        args=training_config,
        callbacks=callbacks,
    )

    # Trainer may expect tokenizer/processing_class depending on trl version
    trainer_params = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = SFTTrainer(**trainer_kwargs)
    trainer.args.disable_tqdm = False

    print(">>> pickle-fix <<<", flush=True)

    # --- pickle-fix V7 block (copied logic from LoRA notebook) ---
    import sys as _sys
    import json as _json
    import trl
    import trl.trainer.sft_config as sft_config_mod
    import trl.trainer.sft_trainer as sft_trainer_mod
    from transformers.trainer import Trainer as HFTrainer
    
    cfg_cls, tr_cls = type(trainer.args), type(trainer)
    cfg_cls.__module__ = "trl.trainer.sft_config"
    tr_cls.__module__ = "trl.trainer.sft_trainer"

    for attr in ("SFTConfig", "UnslothSFTConfig"):
        setattr(sft_config_mod, attr, cfg_cls)
        if hasattr(trl, attr):
            setattr(trl, attr, cfg_cls)

    for attr in ("SFTTrainer", "UnslothSFTTrainer"):
        setattr(sft_trainer_mod, attr, tr_cls)
        if hasattr(trl, attr):
            setattr(trl, attr, tr_cls)

    _sys.modules["trl.trainer.sft_config"] = sft_config_mod
    _sys.modules["trl.trainer.sft_trainer"] = sft_trainer_mod

    def _is_training_args_obj(obj):
        name = type(obj).__name__
        return (
            "SFTConfig" in name
            or "TrainingArguments" in name
            or name in {"SFTConfig", "UnslothSFTConfig", "TrainingArguments", "UnslothTrainingArguments"}
        )

    def _write_args_json(obj, path_hint):
        json_path = str(path_hint).replace(".bin", ".json")
        if not json_path.endswith(".json"):
            json_path = _json_path = (
                os.path.join(str(path_hint), "training_args.json")
                if os.path.isdir(str(path_hint))
                else str(path_hint) + ".json"
            )
        else:
            json_path = json_path

        os.makedirs(os.path.dirname(json_path) or ".", exist_ok=True)
        if hasattr(obj, "to_json_string"):
            with open(json_path, "w", encoding="utf-8") as jf:
                jf.write(obj.to_json_string())
        else:
            payload = obj.to_dict() if hasattr(obj, "to_dict") else {"repr": repr(obj)}
            with open(json_path, "w", encoding="utf-8") as jf:
                _json.dump(payload, jf, ensure_ascii=False, indent=2, default=str)
        print(f"[pickle-fix] wrote {json_path} ({type(obj).__name__})", flush=True)
        return json_path

    _orig_torch_save = torch.save

    def _torch_save_safe(obj, f, *args, **kwargs):
        path = getattr(f, "name", None) or str(f)
        if _is_training_args_obj(obj) or "training_args" in str(path):
            _write_args_json(obj, path)
            return
        return _orig_torch_save(obj, f, *args, **kwargs)

    torch.save = _torch_save_safe
    torch._vinewsqa_pickle_patch = True

    def _install_save_wrapper(cls):
        def _save_fixed(self, output_dir=None, state_dict=None):
            out = output_dir or self.args.output_dir
            os.makedirs(out, exist_ok=True)
            model_to_save = self.model
            if hasattr(self, "accelerator"):
                try:
                    model_to_save = self.accelerator.unwrap_model(self.model)
                except Exception:
                    model_to_save = self.model
            if state_dict is None:
                model_to_save.save_pretrained(out, safe_serialization=True)
            else:
                model_to_save.save_pretrained(out, state_dict=state_dict, safe_serialization=True)
            tok = getattr(self, "processing_class", None) or getattr(self, "tokenizer", None)
            if tok is not None and hasattr(tok, "save_pretrained"):
                tok.save_pretrained(out)
            _write_args_json(self.args, os.path.join(out, "training_args.bin"))

        cls._save = _save_fixed

    _install_save_wrapper(HFTrainer)
    _install_save_wrapper(tr_cls)

    print(f"[pickle-fix] V7 ready | trainer={tr_cls.__name__}", flush=True)

    # --- end pickle-fix block ---

    if _resume_ckpt:
        print(f"[resume] Will resume from {_resume_ckpt}", flush=True)

    try:
        result = trainer.train(resume_from_checkpoint=_resume_ckpt)
        print("Training metrics:", result.metrics)

        best_ckpt = getattr(trainer.state, "best_model_checkpoint", None)
        if best_ckpt and Path(best_ckpt).exists():
            print(f"[fullft] Best dev checkpoint: {best_ckpt}", flush=True)
        else:
            best_ckpt = resolve_best_fullft_checkpoint(output_dir, save_path)

        save_path.mkdir(parents=True, exist_ok=True)
        if Path(best_ckpt).resolve() != save_path.resolve():
            copy_full_model_dir(best_ckpt, save_path)
        else:
            trainer.model.save_pretrained(save_path, safe_serialization=True)
            tokenizer.save_pretrained(save_path)
        print(f"Saved best full model -> {save_path} (from {best_ckpt})")

        if _use_wandb and _wandb_run is not None:
            log_train_summary(_wandb_run, result.metrics)
            # No full model upload: chỉ local.
            upload_dataset_artifact(_wandb_run, PROFILING_CONFIG_PATH)
    finally:
        # Always finish W&B to avoid session leak.
        finish_run(_wandb_run)
        del trainer, model, tokenizer
        clear_gpu()

    return str(save_path)


if RUN_TRAINING:
    trained_fullft_dir = train_fullft_one_run()
    print("Training complete. FullFT dir:", trained_fullft_dir)
else:
    print("RUN_TRAINING=False — skipped.")


In [ ]:
def infer_predictions_subprocess_fullft(model_dir, samples, max_seq_length, log_every):
    import shutil
    import subprocess
    import sys
    import tempfile

    model_dir = Path(model_dir)
    if not model_dir.exists():
        raise FileNotFoundError(model_dir)

    script = Path("eval_infer_subprocess_fullft.py").resolve()
    if not script.exists():
        raise FileNotFoundError(script)

    temp_dir = Path(tempfile.mkdtemp(prefix="vinewsqa_eval_fullft_"))
    samples_path = temp_dir / "samples.json"
    output_path = temp_dir / "predictions.json"

    with open(samples_path, "w", encoding="utf-8") as handle:
        json.dump(samples, handle, ensure_ascii=False)

    command = [
        sys.executable,
        str(script),
        "--model-dir",
        str(model_dir),
        "--samples-json",
        str(samples_path),
        "--output",
        str(output_path),
        "--max-seq-length",
        str(max_seq_length),
        "--max-new-tokens",
        str(MAX_NEW_TOKENS),
        "--batch-size",
        str(INFER_BATCH_SIZE),
        "--log-every",
        str(log_every),
    ]

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line.rstrip(), flush=True)

    return_code = process.wait()
    if return_code:
        shutil.rmtree(temp_dir, ignore_errors=True)
        raise RuntimeError(f"fullft inference failed with exit code {return_code}.")

    with open(output_path, encoding="utf-8") as handle:
        predictions = json.load(handle)

    shutil.rmtree(temp_dir, ignore_errors=True)
    return predictions



def eval_one_fullft_model(model_dir, samples, max_seq_length):
    selected = samples[:]
    predictions = infer_predictions_subprocess_fullft(
        model_dir,
        selected,
        max_seq_length,
        log_every=100,
    )
    if predictions is None:
        return None

    em_values, f1_values = [], []
    for row in predictions:
        em, f1 = score_prediction(row["prediction"], row["gold_answers"])
        row["em"], row["f1"] = em, f1
        em_values.append(em)
        f1_values.append(f1)

    metrics = {
        "method": "fullft",
        "adapter": str(model_dir),
        "exact_match": round(100 * sum(em_values) / len(em_values), 4),
        "f1": round(100 * sum(f1_values) / len(f1_values), 4),
        "n_samples": len(predictions),
    }
    return {"metrics": metrics, "predictions": predictions}


if RUN_METRIC_EVAL:
    # Ưu tiên best checkpoint (tránh eval model overfit cuối run).
    try:
        model_dir = resolve_best_fullft_checkpoint(FULLFT_OUTPUT_DIR, FULLFT_SAVE_PATH)
    except FileNotFoundError:
        model_dir = globals().get("trained_fullft_dir", None) or str(FULLFT_SAVE_PATH)

    print(f"[eval] Evaluating fullft model from: {model_dir}")

    # Smoke test trước khi chạy full test (1987 mẫu)
    SMOKE_EVAL_MAX_SAMPLES = 50
    if SMOKE_EVAL_MAX_SAMPLES and SMOKE_EVAL_MAX_SAMPLES > 0:
        smoke_result = eval_one_fullft_model(
            model_dir,
            eval_samples[:SMOKE_EVAL_MAX_SAMPLES],
            max_seq_length,
        )
        if smoke_result is None:
            raise RuntimeError("Smoke eval returned None.")
        m = smoke_result["metrics"]
        print(
            f"[smoke] EM={m['exact_match']:.2f}% | F1={m['f1']:.2f}% | n={m['n_samples']}",
            flush=True,
        )

    result = eval_one_fullft_model(model_dir, eval_samples, max_seq_length)
    if result is None:
        raise RuntimeError("Eval returned None.")

    fullft_eval_result = result
    metrics = result["metrics"]
    predictions = result["predictions"]

    # Save results json (chỉ fullft)
    with open(RESULTS_FULLFT_TEST_PATH, "w", encoding="utf-8") as handle:
        json.dump(
            {
                "dataset": "ViNewsQA",
                "eval_split": EVAL_SPLIT,
                "expected_test_size": EXPECTED_TEST_SIZE,
                "base_model": BASE_MODEL_NAME,
                "max_seq_length": max_seq_length,
                "summary": metrics,
                "predictions": predictions,
            },
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print("\nFullFT test metrics:")
    print(f"EM={metrics['exact_match']:.2f}% | F1={metrics['f1']:.2f}% | n={metrics['n_samples']}")

    # Optional merge with LoRA comparison file
    merged_out = MERGED_COMPARE_OUT
    if COMPARE_EVAL_PATH.exists():
        with open(COMPARE_EVAL_PATH, encoding="utf-8") as handle:
            comparison = json.load(handle)

        comparison.setdefault("summary", {})["fullft"] = metrics
        comparison.setdefault("predictions", {})["fullft"] = predictions

        with open(merged_out, "w", encoding="utf-8") as handle:
            json.dump(comparison, handle, ensure_ascii=False, indent=2)

        print("Saved merged comparison ->", merged_out)
    else:
        print(f"COMPARE file not found: {COMPARE_EVAL_PATH} (skip merge)")

else:
    print("RUN_METRIC_EVAL=False — skipped.")


## BERTScore evaluation

Reuse FullFT predictions from `fullft_eval_result` / `results_fullft_3b_test_rtx4090.json` / merged compare rtx4090 — no re-inference.
Uses `bert-base-multilingual-cased`; max BERTScore-F1 over gold answers (same as EM/F1).


In [ ]:
# BERTScore for FullFT predictions (no re-infer)
!pip install -q bert-score

FULLFT_BERTSCORE_METHOD = "fullft"
FULLFT_MERGED_COMPARE_PATH = Path(MERGED_COMPARE_OUT)


def load_predictions_for_bertscore():
    """Priority: kernel fullft_eval_result -> RESULTS_FULLFT_TEST_PATH -> merged compare file."""
    results = {}

    mem = globals().get("fullft_eval_result")
    if isinstance(mem, dict) and mem.get("predictions"):
        results[FULLFT_BERTSCORE_METHOD] = {
            "metrics": dict(mem.get("metrics") or {}),
            "predictions": mem["predictions"],
        }
        print(f"[bertscore] Loaded from kernel fullft_eval_result: {sorted(results)}")
        return results

    results_path = Path(RESULTS_FULLFT_TEST_PATH)
    if results_path.is_file():
        with open(results_path, encoding="utf-8") as handle:
            payload = json.load(handle)
        predictions = payload.get("predictions") or []
        if predictions:
            results[FULLFT_BERTSCORE_METHOD] = {
                "metrics": dict(payload.get("summary") or {}),
                "predictions": predictions,
            }
            print(f"[bertscore] Loaded from {results_path}: {sorted(results)}")
            return results

    compare_path = FULLFT_MERGED_COMPARE_PATH
    if compare_path.is_file():
        with open(compare_path, encoding="utf-8") as handle:
            comparison = json.load(handle)
        summary = comparison.get("summary") or {}
        predictions = comparison.get("predictions") or {}
        if FULLFT_BERTSCORE_METHOD in predictions and predictions[FULLFT_BERTSCORE_METHOD]:
            results[FULLFT_BERTSCORE_METHOD] = {
                "metrics": dict(summary.get(FULLFT_BERTSCORE_METHOD) or {}),
                "predictions": predictions[FULLFT_BERTSCORE_METHOD],
            }
            print(f"[bertscore] Loaded from {compare_path}: {sorted(results)}")
            return results

    raise FileNotFoundError(
        "No FullFT predictions for BERTScore. Run RUN_METRIC_EVAL first or provide "
        f"{RESULTS_FULLFT_TEST_PATH} / {FULLFT_MERGED_COMPARE_PATH}"
    )


def compute_bertscore_for_predictions(predictions, model_type=BERTSCORE_MODEL, batch_size=BERTSCORE_BATCH_SIZE):
    """Max BERTScore P/R/F1 over gold_answers per sample; return (per-sample lists, aggregate %)."""
    import torch
    from bert_score import score as bert_score_fn

    cands = [str(row.get("prediction") or "") for row in predictions]
    max_refs = max(len(row.get("gold_answers") or [row.get("ground_truth") or ""]) for row in predictions)
    max_refs = max(max_refs, 1)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    best_p = [-1.0] * len(predictions)
    best_r = [-1.0] * len(predictions)
    best_f = [-1.0] * len(predictions)

    for ref_idx in range(max_refs):
        refs = []
        for row in predictions:
            golds = row.get("gold_answers") or [row.get("ground_truth") or ""]
            if not golds:
                golds = [""]
            refs.append(str(golds[ref_idx] if ref_idx < len(golds) else golds[0]))

        P, R, F1 = bert_score_fn(
            cands,
            refs,
            model_type=model_type,
            device=device,
            batch_size=batch_size,
            verbose=False,
        )
        p_list = P.tolist()
        r_list = R.tolist()
        f_list = F1.tolist()
        for i in range(len(predictions)):
            golds = predictions[i].get("gold_answers") or [predictions[i].get("ground_truth") or ""]
            if not golds:
                golds = [""]
            if ref_idx >= len(golds):
                continue
            if f_list[i] > best_f[i]:
                best_f[i] = f_list[i]
                best_p[i] = p_list[i]
                best_r[i] = r_list[i]

    for i in range(len(predictions)):
        if best_f[i] < 0:
            best_p[i] = best_r[i] = best_f[i] = 0.0

    n = len(predictions)
    agg = {
        "bertscore_precision": round(100 * sum(best_p) / n, 4),
        "bertscore_recall": round(100 * sum(best_r) / n, 4),
        "bertscore_f1": round(100 * sum(best_f) / n, 4),
    }
    return best_p, best_r, best_f, agg


if RUN_BERTSCORE_EVAL:
    bert_results = load_predictions_for_bertscore()
    preds = bert_results[FULLFT_BERTSCORE_METHOD]["predictions"]
    print(f"[bertscore] Scoring {FULLFT_BERTSCORE_METHOD} | n={len(preds)} | model={BERTSCORE_MODEL}", flush=True)
    p_list, r_list, f_list, agg = compute_bertscore_for_predictions(preds)
    for row, p, r, f in zip(preds, p_list, r_list, f_list):
        row["bertscore_p"] = round(float(p), 6)
        row["bertscore_r"] = round(float(r), 6)
        row["bertscore_f1"] = round(float(f), 6)

    metrics = bert_results[FULLFT_BERTSCORE_METHOD].setdefault("metrics", {})
    metrics.update(agg)
    metrics["method"] = FULLFT_BERTSCORE_METHOD
    metrics["n_samples"] = len(preds)
    print(
        f"[bertscore] {FULLFT_BERTSCORE_METHOD}: P={agg['bertscore_precision']:.2f}% "
        f"R={agg['bertscore_recall']:.2f}% F1={agg['bertscore_f1']:.2f}%",
        flush=True,
    )

    print()
    print("=" * 78)
    print(f"ViNewsQA BERTScore — {EVAL_SPLIT} | model={BERTSCORE_MODEL}")
    print(f"{'Method':<12} {'EM':>10} {'Token-F1':>10} {'BERT-P':>10} {'BERT-R':>10} {'BERT-F1':>10} {'Samples':>10}")
    print("-" * 78)
    m = metrics
    em = m.get("exact_match")
    tok = m.get("f1")
    em_s = f"{em:.2f}%" if isinstance(em, (int, float)) else "—"
    tok_s = f"{tok:.2f}%" if isinstance(tok, (int, float)) else "—"
    print(
        f"{FULLFT_BERTSCORE_METHOD:<12} {em_s:>10} {tok_s:>10} "
        f"{m.get('bertscore_precision', 0):>9.2f}% "
        f"{m.get('bertscore_recall', 0):>9.2f}% "
        f"{m.get('bertscore_f1', 0):>9.2f}% "
        f"{m.get('n_samples', 0):>10}"
    )
    print("=" * 78)

    out_payload = {
        "dataset": "ViNewsQA",
        "format": "SQuAD 1.1 extractive QA",
        "eval_split": EVAL_SPLIT,
        "expected_test_size": EXPECTED_TEST_SIZE,
        "base_model": BASE_MODEL_NAME,
        "bertscore_model": BERTSCORE_MODEL,
        "summary": {FULLFT_BERTSCORE_METHOD: metrics},
        "predictions": {FULLFT_BERTSCORE_METHOD: preds},
    }

    compare_path = FULLFT_MERGED_COMPARE_PATH
    if compare_path.is_file():
        with open(compare_path, encoding="utf-8") as handle:
            old = json.load(handle)
        old_metrics = (old.get("summary") or {}).get(FULLFT_BERTSCORE_METHOD) or {}
        for key in ("exact_match", "f1", "adapter"):
            if key in old_metrics and key not in out_payload["summary"][FULLFT_BERTSCORE_METHOD]:
                out_payload["summary"][FULLFT_BERTSCORE_METHOD][key] = old_metrics[key]

    with open(BERTSCORE_OUT_PATH, "w", encoding="utf-8") as handle:
        json.dump(out_payload, handle, ensure_ascii=False, indent=2)
    print("Saved BERTScore comparison ->", BERTSCORE_OUT_PATH)
else:
    print("RUN_BERTSCORE_EVAL=False — skipped.")
